Strategia inwestycyjna (decyzje wejścia i wyjścia) dla spółki  Apple (AAPL), test w okresie od 01.01.2024 - 06.05.2024

In [31]:
pip install ta

In [32]:
import pandas as pd
import numpy as np
import yfinance as yf
from ta.momentum import RSIIndicator
from ta.trend import SMAIndicator
from ta.volatility import AverageTrueRange

# Pobranie danych giełdowych AAPL od 2020 roku
ticker = "AAPL"
start = "2020-01-01"
end   = "2024-05-07"

data = yf.download(ticker, start=start, end=end)
data.dropna(inplace=True)

data.columns = data.columns.get_level_values(0)
data.columns = ["Open", "High", "Low", "Close", "Volume"]

#Wskaźniki
# RSI - momentum
rsi_period = 14
data["RSI"] = RSIIndicator(close=data["Close"], window=rsi_period).rsi()

# SMA_20 - śr. krocząca
sma_window = 20
data["SMA_20"] = SMAIndicator(close=data["Close"], window=sma_window).sma_indicator()

# ATR - do SL/TP
atr_period = 14
atr = AverageTrueRange(
    high=data["High"],
    low=data["Low"],
    close=data["Close"],
    window=atr_period
)
data["ATR"] = atr.average_true_range()

# Lag1 Close - opóźnienie cen close
data["Close_lag1"] = data["Close"].shift(1)


data.dropna(inplace=True)
print(data.columns)

data = data.copy()
data.columns = [str(c) for c in data.columns]  # bez multindexu

data

/tmp/ipython-input-3640799297.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start, end=end)
[*********************100%***********************]  1 of 1 completed

Index(['Open', 'High', 'Low', 'Close', 'Volume', 'RSI', 'SMA_20', 'ATR',
       'Close_lag1'],
      dtype='object')


,Open,High,Low,Close,Volume,RSI,SMA_20,ATR,Close_lag1
Date,,,,,,,,,
2020-01-30,78.143158,78.196240,76.907809,77.339701,126743200,62.969015,74.961404,1.731942,78.283113
2020-01-31,74.678375,77.856020,74.384014,77.433781,199588400,63.311235,75.259262,1.856232,77.339701
2020-02-03,74.473312,75.638689,72.919472,73.421330,173788400,44.445859,75.345521,2.046095,77.433781
2020-02-04,76.931938,77.122551,75.672459,76.077807,136616400,54.180427,75.605138,2.164318,73.421330
2020-02-05,77.559265,78.357899,76.956067,78.058708,118826800,59.832599,75.890813,2.172588,76.077807
...,...,...,...,...,...,...,...,...,...
2024-04-30,168.951782,173.574080,168.624451,171.927508,65934800,56.272437,168.241566,3.889340,171.967177
2024-05-01,167.930115,171.312526,167.741650,168.207848,50383100,47.385021,168.280746,3.910520,171.927508
2024-05-02,171.629929,172.016772,169.507245,171.114132,94214900,53.556939,168.390850,3.903263,168.207848


In [33]:
from prophet import Prophet
from sklearn.metrics import mean_absolute_error
from itertools import product


In [34]:

# Przygotowanie danych dla Prophet
df = data.copy()
df_prophet = pd.DataFrame({
    "ds": df.index,
    "y": df["Close"],
    "RSI": df["RSI"],
    "SMA_20": df["SMA_20"],
    "ATR": df["ATR"],
    "Close_lag1": df["Close_lag1"]
})
df_prophet["ds"] = pd.to_datetime(df_prophet["ds"])
print(df_prophet)

# Podział na zbiór test i train
train_df = df_prophet[df_prophet["ds"] < "2024-01-01"]
test_df_prophet = df_prophet[df_prophet["ds"] >= "2024-01-01"]


# Siatka hiperparametrów
cp_scales = [0.01, 0.05, 0.1]
season_scales = [1.0, 5.0, 10.0]

best_params = None
best_mae = np.inf

#Przeszukiwanie siatki hiperparametrów
for cp, ss in product(cp_scales, season_scales):
    m = Prophet(
        changepoint_prior_scale=cp,
        seasonality_prior_scale=ss,
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=False
    )
    # dodatek regresorów
    for reg in ["RSI", "SMA_20", "ATR", "Close_lag1"]:
        m.add_regressor(reg)

#trenowanie na danych hist
    m.fit(train_df)

    future = train_df[["ds", "RSI", "SMA_20", "ATR", "Close_lag1"]].copy()
    forecast = m.predict(future)

#MAE
    mae = mean_absolute_error(train_df["y"], forecast["yhat"])
    if mae < best_mae:
        best_mae = mae
        best_params = {"changepoint_prior_scale": cp, "seasonality_prior_scale": ss}

print("Najlepsze parametry:", best_params, "MAE:", best_mae)


                   ds           y        RSI      SMA_20       ATR  Close_lag1
Date                                                                          
2020-01-30 2020-01-30   77.339701  62.969015   74.961404  1.731942   78.283113
2020-01-31 2020-01-31   77.433781  63.311235   75.259262  1.856232   77.339701
2020-02-03 2020-02-03   73.421330  44.445859   75.345521  2.046095   77.433781
2020-02-04 2020-02-04   76.077807  54.180427   75.605138  2.164318   73.421330
2020-02-05 2020-02-05   78.058708  59.832599   75.890813  2.172588   76.077807
...               ...         ...        ...         ...       ...         ...
2024-04-30 2024-04-30  171.927508  56.272437  168.241566  3.889340  171.967177
2024-05-01 2024-05-01  168.207848  47.385021  168.280746  3.910520  171.927508
2024-05-02 2024-05-02  171.114132  53.556939  168.390850  3.903263  168.207848
2024-05-03 2024-05-03  185.139725  71.147055  169.236948  4.651085  171.114132
2024-05-06 2024-05-06  180.874536  63.296036  169.89

In [35]:
#model z najlepszymi hiperparametrami
m_best = Prophet(
    changepoint_prior_scale=best_params["changepoint_prior_scale"],
    seasonality_prior_scale=best_params["seasonality_prior_scale"],
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=False
)
for reg in ["RSI", "SMA_20", "ATR", "Close_lag1"]:
    m_best.add_regressor(reg)

m_best.fit(train_df)

# Przygotowanie danych testowych
future_test = test_df_prophet[["ds", "RSI", "SMA_20", "ATR", "Close_lag1"]]
forecast_test = m_best.predict(future_test)
df.loc[test_df_prophet["ds"], "yhat"] = forecast_test["yhat"].values


In [36]:
threshold = 0.001

df["yhat_shift1"] = df["yhat"].shift(1)  # prognoza na dzisiejszy dzień wygenerowana wczoraj
df["signal"] = 0

#wejście w longa
long_cond = (
    (df["yhat_shift1"] > df["Close"] * (1 + threshold)) &
    (df["Close"] > df["SMA_20"]) &
    (df["RSI"] < 70)
)

df.loc[long_cond, "signal"] = 1


In [37]:
test_start = "2023-12-31"
test_end   = "2024-05-07"
test_df = df.loc[test_start:test_end].copy()
test_df

,Open,High,Low,Close,Volume,RSI,SMA_20,ATR,Close_lag1,yhat,yhat_shift1,signal
Date,,,,,,,,,,,,
2024-01-02,183.903214,186.677021,182.169586,185.399081,82488700,37.980915,192.235540,3.321015,192.085938,186.636768,NaN,0
2024-01-03,182.526230,184.140985,181.713894,182.496512,58414500,33.070408,191.950235,3.347028,185.399081,182.029779,186.636768,0
2024-01-04,180.208130,181.377083,179.187767,180.445875,71983600,30.108690,191.551004,3.344293,182.496512,179.574065,182.029779,0
2024-01-05,179.484955,181.050175,178.484409,180.287390,62379700,29.885921,190.933833,3.288684,180.445875,178.449483,179.574065,0
2024-01-08,183.823975,183.863609,179.801961,180.386437,59144500,30.233338,190.362231,3.343896,180.287390,178.322141,178.449483,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2024-04-30,168.951782,173.574080,168.624451,171.927508,65934800,56.272437,168.241566,3.889340,171.967177,175.990876,174.291198,1
2024-05-01,167.930115,171.312526,167.741650,168.207848,50383100,47.385021,168.280746,3.910520,171.927508,172.515757,175.990876,0
2024-05-02,171.629929,172.016772,169.507245,171.114132,94214900,53.556939,168.390850,3.903263,168.207848,173.381631,172.515757,1


In [38]:
pip install backtesting

In [39]:
# TRAIN: 2020–2023
train_data = data.loc['2020-01-01':'2023-12-31'].copy()
train_data['Prediction'] = df.loc['2020-01-01':'2023-12-31', 'yhat']

# TEST: 2024
test_data = data.loc['2024-01-01':'2024-12-31'].copy()
test_data['Prediction'] = df.loc['2024-01-01':'2024-12-31', 'yhat']


In [40]:
from backtesting import Strategy, Backtest
class HighWinRateStrategyy(Strategy):
    tp_atr = 0.6
    sl_atr = 1.2
    threshold = 0.001
    rsi_upper = 60
    rsi_lower = 40
    atr_filter = 0.7

    def init(self):
      self.pred = self.I(lambda: self.data.Prediction)
      self.atr = self.I(lambda: self.data.ATR)
      self.rsi = self.I(lambda: self.data.RSI)
      self.atr_median = np.median(self.data.ATR)

    def next(self):
        price = self.data.Close[-1]
        vol = self.atr[-1]

        #brak handlu przy niskim ATR
        if vol < self.atr_median * self.atr_filter:
            return

        # Rsi ok. 50 - nie handluj
        if 45 < self.rsi[-1] < 55:
            return

        # ignoruj płaskie sygnały
        if abs(self.pred[-1] - price) < price * 0.001:
            return

        sl_dist = vol * self.sl_atr
        tp_dist = vol * self.tp_atr

        signal_long = (
            self.pred[-1] > price * (1 + self.threshold)
            and self.rsi[-1] < self.rsi_upper
        )

        signal_short = (
            self.pred[-1] < price * (1 - self.threshold)
            and self.rsi[-1] > self.rsi_lower
        )

        #wyjście
        if signal_short and self.position.is_long:
            self.position.close()
        if signal_long and self.position.is_short:
            self.position.close()

        # Wejścia
        if signal_long and not self.position:
            self.buy(sl=price - sl_dist, tp=price + tp_dist)
        elif signal_short and not self.position:
            self.sell(sl=price + sl_dist, tp=price - tp_dist)


In [41]:
bt_train = Backtest(train_data, HighWinRateStrategyy, cash=10000, commission=0.001)

stats_train, heatmap_train = bt_train.optimize(
    tp_atr=[0.4, 0.5, 0.6, 0.7, 0.8],
    sl_atr=[1.0, 1.2, 1.4],
    threshold=[0.0010, 0.0015, 0.0020, 0.0025],
    rsi_upper=[45, 50, 55, 60],
    rsi_lower=[30, 35, 40],
    atr_filter=[0.5, 0.6, 0.7, 0.8],

    maximize='Return [%]',
    constraint=lambda p: p.tp_atr <= p.sl_atr * 1.5,
    return_heatmap=True
)

print("\n--- NAJLEPSZE PARAMETRY (TRAIN) ---")
print(stats_train._strategy)


/usr/local/lib/python3.12/dist-packages/backtesting/backtesting.py:1624: UserWarning: Searching for best of 2880 configurations.
  output = _optimize_grid()


Backtest.optimize:   0%|          | 0/2880 [00:00<?, ?it/s]

Backtest.run:   0%|          | 0/986 [00:00<?, ?bar/s]


--- NAJLEPSZE PARAMETRY (TRAIN) ---
HighWinRateStrategyy(tp_atr=0.4,sl_atr=1.0,threshold=0.001,rsi_upper=45,rsi_lower=30,atr_filter=0.5)


In [42]:
best_params = stats_train._strategy

bt_test = Backtest(test_data, HighWinRateStrategyy, cash=10000, commission=0.001)

strategy_params = {
    'tp_atr': best_params.tp_atr,
    'sl_atr': best_params.sl_atr,
    'threshold': best_params.threshold,
    'rsi_upper': best_params.rsi_upper,
    'rsi_lower': best_params.rsi_lower,
    'atr_filter': best_params.atr_filter
}

stats_test = bt_test.run(**strategy_params)

print("wyniki na test")
print(stats_test)

Backtest.run:   0%|          | 0/86 [00:00<?, ?bar/s]

wyniki na test
Start                     2024-01-02 00:00:00
End                       2024-05-06 00:00:00
Duration                    125 days 00:00:00
Exposure Time [%]                    59.77011
Equity Final [$]                  10887.16099
Equity Peak [$]                   11090.39552
Commissions [$]                     560.03984
Return [%]                            8.87161
Buy & Hold Return [%]                -2.44044
Return (Ann.) [%]                    27.91609
Volatility (Ann.) [%]                16.94133
CAGR [%]                             18.69158
Sharpe Ratio                          1.64781
Sortino Ratio                         4.29785
Calmar Ratio                          8.02579
Alpha [%]                             9.02777
Beta                                  0.06399
Max. Drawdown [%]                     -3.4783
Avg. Drawdown [%]                     -1.2238
Max. Drawdown Duration       32 days 00:00:00
Avg. Drawdown Duration       11 days 00:00:00
# Trades           

W badanym okresie strategia osiągnęła stopę zwrotu +8.87%, podczas gdy proste kup i trzymaj (Buy & Hold) dla AAPL wygenerowało wynik –2.44%. Oznacza to, że strategia pokonała rynek o ponad 11 punktów procentowych, mimo że okres testowy obejmował fragment rynku o ograniczonym trendzie wzrostowym.

Strategia wykonała 27 transakcji, co odpowiada jej charakterowi swingowemu. Nie jest nadmiernie aktywna, ale reaguje na wyraźne sygnały z modelu predykcyjnego i wskaźników technicznych.

Kluczowym atutem strategii jest wysoki poziom bezpieczeństwa kapitału, potwierdzony niskim maksymalnym obsunięciem (Max Drawdown -3.48%) oraz wysokim wskaźnikiem Profit Factor (2.23).

In [43]:
bt_test.plot()

GridPlot(id='p2367', ...)